In [5]:
import pandas as pd
import re

def keep_earliest_result_per_patient(df, question_col="Question", date_label="Date"):
    """
    Keep one column per patient (BW_###), choosing the column with the earliest Date value.
    Returns a cleaned DataFrame with:
    - question_col
    - one selected column per patient
    """
    if question_col not in df.columns:
        raise ValueError(f"Missing required column: {question_col}")

    # Find the row that contains dates
    date_rows = df.index[df[question_col].astype(str).str.strip().eq(date_label)]
    if len(date_rows) == 0:
        raise ValueError(f"No row found where {question_col} == {date_label}")
    date_row = date_rows[0]

    # Candidate patient columns
    patient_cols = [c for c in df.columns if re.match(r"^BW_\d{3}(?:\.\d+)?$", str(c))]
    if not patient_cols:
        raise ValueError("No patient columns found matching BW_### or BW_###.n")

    # Group duplicate columns by base patient ID
    grouped = {}
    for col in patient_cols:
        base_id = str(col).split(".")[0]
        grouped.setdefault(base_id, []).append(col)

    keep_cols = [question_col]

    for base_id, cols in grouped.items():
        # Convert the Date row values for these columns to datetime
        dates = pd.to_datetime(df.loc[date_row, cols], errors="coerce")

        # If all date values are invalid/NaT, keep first by stable order
        if dates.isna().all():
            chosen_col = sorted(cols)[0]
        else:
            # Earliest valid date
            chosen_col = dates.idxmin()

        keep_cols.append(chosen_col)

    out = df.loc[:, keep_cols].copy()

    # Rename selected patient columns to base ID (remove .1/.2 suffixes)
    rename_map = {c: str(c).split(".")[0] for c in out.columns if c != question_col}
    out = out.rename(columns=rename_map)

    return out


def save_dataframe_to_csv(df, output_csv_path, index=False):
    """
    Save DataFrame to CSV.
    """
    df.to_csv(output_csv_path, index=index)
    print(f"Saved CSV to: {output_csv_path}")

In [6]:
"""
Generate a csv for each patient column, and save in the same directory as their MRI data. Only for patients that already exist in the directory
Inputs:
df: DataFrame containing patient columns (e.g. BW_ID) and a "Question" column
data_root: Path to the root directory containing patient subdirectories
Output:
For each patient column in df, saves a CSV file with "Question" and "Score" columns in the corresponding patient directory under data_root.
"""
from pathlib import Path

def save_patient_csvs(df: pd.DataFrame, data_root: Path):
    question_col = "Question"
    patient_cols = [c for c in df.columns if c != question_col]
    count = 0

    for col in patient_cols:
        patient_id = col
        patient_dir = data_root/patient_id
        # patient_dir_id = str(patient_id).replace("_", "-")  # Convert BW_003 to BW-003 for directory name
        # patient_dir = data_root/patient_dir_id
        if not patient_dir.exists():
            print(f"Directory for patient {patient_id} does not exist.")
            continue
        output_csv_path = patient_dir/"eortc_scores.csv"

        # Create a DataFrame with Question and the patient's column
        patient_df = df[[question_col, col]].copy()
        patient_df = patient_df.rename(columns={col: "Score"})

        # Save to CSV
        save_dataframe_to_csv(patient_df, output_csv_path, index=False)
        count += 1

    print(f"Saved CSV files for {count} patients out of {len(patient_cols)}.")

In [7]:
"""
Generate pandas dataframe containing different scores for each patient in the Brainwear dataset.
Takes excel file as input and outputs two dataframes: one with raw scores and one with calculated scores.
"""
import sys
from pathlib import Path

cwd = Path.cwd() # /path/to/home
PROJECT_ROOT = Path("/path/to/BrainWear_Kareem/")
FYP_ROOT = PROJECT_ROOT/"FYP"

if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))
print(f"Added to path: {FYP_ROOT}")

from utils.eortc import load_eortc


raw_df, score_df = load_eortc(PROJECT_ROOT, "brainwear_data_nhs.xlsx")

# print(raw_df.head())
# print(score_df.head())

earliest_df = keep_earliest_result_per_patient(score_df)
# print(earliest_df.head())

save_dataframe_to_csv(earliest_df, PROJECT_ROOT/"eortc_scores.csv")

save_patient_csvs(earliest_df, PROJECT_ROOT/"Processed_Brainwear")